<a href="https://colab.research.google.com/github/mariangelesalomar-sudo/eigenfaces-dma-grupo-1/blob/main/GRUPO_1__zBackPropagation_multiclase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmo de BackPropagation multiclase

Este codigo se utilizará para entrenar la red neuronal de clasificacion de Nuestras Caras



### Código en Python del Algoritmo de BackPropagation

La clase del dataset debe ser categorica
Existe una clase en Python que resuelve el problema

In [ ]:
# conexion al Google Drive
from google.colab import drive
drive.mount('/content/.drive')
!mkdir -p "/content/.drive/My Drive/DMA"
!mkdir -p "/content/buckets"
!ln -s "/content/.drive/My Drive/DMA" /content/buckets/b1

Mounted at /content/.drive


In [ ]:
# instalo  itables solo si no esta instalado
!pip show itables >/dev/null || pip install itables

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.4 MB/s eta 0:00:00


In [ ]:
import polars as pl
import numpy as np
import math
import matplotlib.pyplot as plt
%matplotlib inline
from IPython import display
import time
import os
import pickle
from functools import reduce
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)


In [ ]:
# definicion de la clase de graficos
# ADAPTACION: guard para datos con mas de 2 dimensiones
# (nuestro caso: 80 componentes PCA/ISOMAP)
# En ese caso se imprime epoch y MSE por consola en vez de graficar
# las rectas de decision, que solo tienen sentido en 2D.

class perceptron_plot:
    """plotting first hidden layer class"""
    def __init__(self, X, Y, delay) -> None:
        self.X = X
        self.Y = Y
        self.delay = delay
        self.es_2d = (X.shape[1] == 2)

        if not self.es_2d:
            return  # no inicializar grafico si >2D

        x1_min = np.min(X[:,0])
        x2_min = np.min(X[:,1])
        x1_max = np.max(X[:,0])
        x2_max = np.max(X[:,1])
        self.x1_min = x1_min - 0.1*(x1_max - x1_min)
        self.x1_max = x1_max + 0.1*(x1_max - x1_min)
        self.x2_min = x2_min - 0.1*(x2_max - x2_min)
        self.x2_max = x2_max + 0.1*(x2_max - x2_min)
        self.fig = plt.figure(figsize = (10,8))
        self.ax = self.fig.subplots()
        self.ax.set_xlim(self.x1_min, self.x1_max, auto=False)
        self.ax.set_ylim(self.x2_min, self.x2_max, auto=False)

    def graficarVarias(self, W, x0, epoch, error) -> None:
        if not self.es_2d:
            # Con 80 componentes no tiene sentido graficar rectas de decision
            # Se imprime el progreso por consola
            display.clear_output(wait=True)
            print(f"epoch {epoch:>6d}  |  MSE {error:.6f}")
            return

        display.clear_output(wait=True)
        plt.cla()
        self.ax.set_xlim(self.x1_min, self.x1_max)
        self.ax.set_ylim(self.x2_min, self.x2_max)
        plt.title('epoch ' + str(epoch) + '  reg ' + "{0:.2E}".format(error))
        scatter = self.ax.scatter(self.X[:,0], self.X[:,1], c=self.Y, s=20)
        for i in range(len(x0)):
            vx2_min = -(W[i,0]*self.x1_min + x0[i])/W[i,1]
            vx2_max = -(W[i,0]*self.x1_max + x0[i])/W[i,1]
            self.ax.plot([self.x1_min, self.x1_max],
                         [vx2_min, vx2_max],
                         linewidth=2, color='red', alpha=0.5)
        display.display(plt.gcf())
        time.sleep(self.delay)

In [ ]:
# definicion de las funciones de activacion
#  y sus derivadas
#  ahora agregando las versiones VECTORIZADAS

def func_eval(fname, x):
    match fname:
        case "purelin":
            y = x
        case "logsig":
            y = 1.0 / ( 1.0 + math.exp(-x) )
        case "tansig":
            y = 2.0 / ( 1.0 + math.exp(-2.0*x) ) - 1.0
    return y

# version vectorizada de func_eval
func_eval_vec = np.vectorize(func_eval)


def deriv_eval(fname, y):  #atencion que y es la entrada y=f( x )
    match fname:
        case "purelin":
            d = 1.0
        case "logsig":
            d = y*(1.0-y)
        case "tansig":
            d = 1.0 - y*y
    return d


# version vectorizada de deriv_eval
deriv_eval_vec = np.vectorize(deriv_eval)

### Clase  multiperceptron
entrenar, predecir

In [ ]:
# definicion de la clase de multiperceptron
# FIXES aplicados respecto al original del profe:
#   1. entrenar(): carpeta → self.carpeta en el pickle (evita NameError)
#   2. predecir(): epoch y MSE tomados de self.red['work'] (evita NameError)
#   3. predecir(): se salta el grafico si los datos tienen mas de 2 dimensiones
#   4. predecir(): columna clase hardcodeada a "y" → usa el parametro clase

class multiperceptron(object):
    """Multiperceptron class"""

    # inicializacion de los pesos de todas las capas
    def _red_init(self, semilla) -> None:
        niveles = self.red['arq']['layers_qty']
        np.random.seed(semilla)
        for i in range(niveles):
           nivel = dict()
           nivel['id'] = i
           nivel['last'] = (i==(niveles-1))
           nivel['size'] = self.red["arq"]["layers_size"][i]
           nivel['func'] = self.red["arq"]["layers_func"][i]
           if( i==0 ):
              entrada_size = self.red['arq']['input_size']
           else:
              entrada_size = self.red['arq']['layers_size'][i-1]
           salida_size = nivel['size']
           nivel['W'] = np.random.uniform(-0.5, 0.5, [salida_size, entrada_size])
           nivel['w0'] = np.random.uniform(-0.5, 0.5, [salida_size, 1])
           nivel['W_m'] = np.zeros([salida_size, entrada_size])
           nivel['w0_m'] = np.zeros([salida_size, 1])
           self.red['layer'].append(nivel)

    # constructor generico
    def __init__(self) -> None:
        self.data = dict()
        self.red = dict()
        self.carpeta = ""

    # inicializacion full
    def inicializar(self, df, campos, clase, hidden_layers_sizes, layers_func,
                 semilla, carpeta) -> None:
        self.data['X'] = np.array( df.select(campos))
        X_mean = self.data['X'].mean(axis=0)
        X_sd = self.data['X'].std(axis=0)
        self.data['X'] = (self.data['X'] - X_mean)/X_sd
        label = df.select(clase)
        self.data['Ylabel'] = np.array(label).reshape(len(label))
        col_originales = df.columns
        self.data['Y'] = np.array( df.to_dummies(clase).drop(col_originales, strict=False) )
        col_dummies = sorted( list( set(df.to_dummies(clase).columns) - set(col_originales)))
        clases_originales = reduce(lambda acc, x: acc + [x[(len(clase)+1):]], col_dummies, [])
        tamanos = hidden_layers_sizes
        tamanos.append(self.data['Y'].shape[1])
        arquitectura = {
             'input_size' : self.data['X'].shape[1],
             'input_mean' : X_mean,
             'input_sd'   : X_sd,
             'output_values' : clases_originales,
             'layers_qty' : len(hidden_layers_sizes),
             'layers_size': tamanos,
             'layers_func': layers_func,
        }
        self.red['arq'] = arquitectura
        self.red['work'] = dict()
        self.red['work']['epoch'] = 0
        self.red['work']['MSE'] = float('inf')
        self.red['work']['train_error_rate'] = float('inf')
        self.red['layer'] = list()
        self._red_init(semilla)
        self.carpeta = carpeta
        os.makedirs(self.carpeta, exist_ok=True)
        with open(self.carpeta+"/data.pkl", 'wb') as f:
            pickle.dump(self.data, f)
        with open(self.carpeta+"/red.pkl", 'wb') as f:
            pickle.dump(self.red, f)

    # Algoritmo Backpropagation
    def entrenar(self, epoch_limit, MSE_umbral,
               learning_rate, lr_momento, save_frequency,
               retomar=True) -> None:
        if retomar:
            with open(self.carpeta+"/data.pkl", 'rb') as f:
              self.data = pickle.load(f)
            with open(self.carpeta+"/red.pkl", 'rb') as f:
              self.red = pickle.load(f)
        epoch = self.red['work']['epoch']
        MSE = self.red['work']['MSE']
        grafico = perceptron_plot(X=self.data['X'], Y=self.data['Ylabel'], delay=0.1)
        Xfilas = self.data['X'].shape[0]
        niveles = self.red["arq"]["layers_qty"]

        while ( MSE > MSE_umbral) and (epoch < epoch_limit):
          epoch += 1
          for fila in range(Xfilas):
             x = self.data['X'][fila:fila+1,:]
             clase = self.data['Y'][fila:fila+1,:]
             entrada = x.T
             vsalida = [0] * niveles
             for i in range(niveles):
               estimulos = self.red['layer'][i]['W'] @ entrada + self.red['layer'][i]['w0']
               vsalida[i] = func_eval_vec(self.red['layer'][i]['func'], estimulos)
               entrada = vsalida[i]
             verror = [0] * (niveles+1)
             verror[niveles] = clase.T - vsalida[niveles-1]
             i = niveles-1
             verror[i] = verror[i+1] * deriv_eval_vec(self.red['layer'][i]['func'], vsalida[i])
             for i in reversed(range(niveles-1)):
               verror[i] = deriv_eval_vec(self.red['layer'][i]['func'], vsalida[i]) * (self.red['layer'][i+1]['W'].T @ verror[i+1])
             entrada = x.T
             for i in range(niveles):
               self.red['layer'][i]['W_m'] = learning_rate * (verror[i] @ entrada.T) + lr_momento * self.red['layer'][i]['W_m']
               self.red['layer'][i]['w0_m'] = learning_rate * verror[i] + lr_momento * self.red['layer'][i]['w0_m']
               self.red['layer'][i]['W'] = self.red['layer'][i]['W'] + self.red['layer'][i]['W_m']
               self.red['layer'][i]['w0'] = self.red['layer'][i]['w0'] + self.red['layer'][i]['w0_m']
               entrada = vsalida[i]

          entrada = self.data['X'].T
          for i in range(niveles):
            estimulos = self.red['layer'][i]['W'] @ entrada + self.red['layer'][i]['w0']
            salida = func_eval_vec(self.red['layer'][i]['func'], estimulos)
            entrada = salida

          MSE = np.mean( (self.data['Y'].T - salida)**2 )

          if (epoch % save_frequency == 0) or (MSE <= MSE_umbral) or (epoch >= epoch_limit):
              W = self.red['layer'][0]['W']
              w0 = self.red['layer'][0]['w0']
              grafico.graficarVarias(W, w0.T[0], epoch, MSE)
              self.red['work']['epoch'] = epoch
              self.red['work']['MSE'] = MSE
              prediccion = np.argmax(salida.T, axis=1)
              out = np.array(self.red["arq"]['output_values'])
              error_rate = np.mean(self.data['Ylabel'] != out[prediccion])
              self.red["work"]['train_error_rate'] = error_rate
              # FIX: self.carpeta en vez de carpeta (variable suelta)
              with open(self.carpeta+"/red.pkl", 'wb') as f:
                 pickle.dump(self.red, f)

        return (epoch, MSE, self.red['work']['train_error_rate'])

    # predigo a partir de modelo recien entrenado
    def predecir(self, df_new, campos, clase) -> None:
        niveles = self.red['arq']['layers_qty']
        X_new = np.array(df_new.select(campos))
        X_new = (X_new - self.red['arq']['input_mean']) / self.red['arq']['input_sd']

        # FIX: solo graficar si los datos son 2D
        if X_new.shape[1] == 2:
            Ylabel_new = df_new.select(clase)
            Ylabel_new = np.array(Ylabel_new).reshape(len(Ylabel_new))
            grafico = perceptron_plot(X=X_new, Y=Ylabel_new, delay=0.1)
            W = self.red['layer'][0]['W']
            w0 = self.red['layer'][0]['w0']
            # FIX: epoch y MSE desde self.red['work'], no variables sueltas
            epoch_g = self.red['work']['epoch']
            MSE_g   = self.red['work']['MSE']
            grafico.graficarVarias(W, w0.T[0], epoch_g, MSE_g)

        entrada = X_new.T
        for i in range(niveles):
          estimulos = self.red['layer'][i]['W'] @ entrada + self.red['layer'][i]['w0']
          salida = func_eval_vec(self.red['layer'][i]['func'], estimulos)
          entrada = salida

        pred_idx = np.argmax(salida.T, axis=1)
        pred_raw = np.max(salida.T, axis=1)
        out = np.array(self.red['arq']['output_values'])
        # FIX: usa el parametro clase, no el string hardcodeado "y"
        Ylabel_new_arr = np.array(df_new.select(clase)).flatten()
        error_rate = np.mean(Ylabel_new_arr != out[pred_idx])

        return (out[pred_idx], pred_raw, error_rate)

    # cargo un modelo ya entrenado, grabado en carpeta
    def cargar_modelo(self, carpeta) -> None:
        self.carpeta = carpeta
        with open(self.carpeta+"/red.pkl", 'rb') as f:
          self.red = pickle.load(f)
        return (self.red['work']['epoch'],
                self.red['work']['MSE'],
                self.red['work']['train_error_rate'])

## 1 Lectura del Dataset

En este humilde y restringida version, la clase del dataset debe ser categorica, no es capaz de trabajar con clases continuas.
La clase categorica puede ser  n-aria

In [ ]:
# Lectura del dataset con Polars  (adaptado de arrays numpy)

# ============================================================
# CONFIGURACION — ajustar estos dos parametros
# ============================================================
MODO           = "isomap"     # <<< CAMBIAR A "isomap" PARA LA SEGUNDA RED
CARPETA_DATOS  = "/content/buckets/b1/datos_30x30"
carpeta        = f"/content/buckets/b1/nn_{MODO}/"  # donde se guarda el modelo
# ============================================================

if MODO == "pca":
    X_arr = np.load(f"{CARPETA_DATOS}/fotos_proyectadas_pca.npy")
elif MODO == "isomap":
    X_arr = np.load(f"{CARPETA_DATOS}/fotos_proyectadas_isomap.npy")
else:
    raise ValueError("MODO debe ser 'pca' o 'isomap'")

etiquetas_arr = np.load(f"{CARPETA_DATOS}/etiquetas.npy", allow_pickle=True)

n_componentes = X_arr.shape[1]  # 80
campos = [f"x{i+1}" for i in range(n_componentes)]  # x1, x2, ..., x80

data_dict = {campos[i]: X_arr[:, i].astype(np.float64) for i in range(n_componentes)}
data_dict["y"] = etiquetas_arr.astype(str)

df = pl.DataFrame(data_dict)

print(f"Modo: {MODO}")
print(f"Fotos: {df.shape[0]}  |  Componentes: {n_componentes}")
print(f"Alumnos: {sorted(df['y'].unique().to_list())}")
df

Modo: isomap
Fotos: 2068  |  Componentes: 80
Alumnos: ['agustina_sebben', 'belen_maldonado', 'fede_spinelli', 'guillermo_anso', 'juani_cacchione', 'juani_paberolis', 'judi_luna', 'lucia_tamplin', 'mariangeles_alomar', 'martin_ceriotti', 'matias_villanueva', 'miguel_garrone', 'millie_teran', 'tomas_delbo']


Loading ITables v2.7.3 from the internet... (need help?)


### 1.1 Particion  training/testing



*   Es valido cambiar la *semilla_particion* para probar distintos <test, train> y asi estimar con mas precisión el error rate en testing  (Montecarlo Estimation)



In [ ]:
# particion del dataset en training/testing

semilla_particion = 26011989
pct_train = 0.75  # ratio de registros que va a training


def train_test_split_df(df, seed=0, test_size=0.2):
    return df.with_columns(
        pl.int_range(pl.len(), dtype=pl.UInt32)
        .shuffle(seed=seed)
        .gt(pl.len() * test_size)
        .alias("split")
    ).partition_by("split", include_key=False)


(df_train, df_test) = train_test_split_df(df,
                                          seed=semilla_particion,
                                          test_size=pct_train)

# imprimo los tamaños
print("Train:", df_train.shape)
print("Test:", df_test.shape)

# verifico que todas las clases esten en ambos conjuntos
clases_train = set(df_train['y'].unique().to_list())
clases_test  = set(df_test['y'].unique().to_list())
faltantes    = clases_test - clases_train
if faltantes:
    print(f"⚠️  ALERTA: clases en test pero NO en train: {faltantes}")
else:
    print("✓ Todas las clases presentes en ambos conjuntos")

Train: (1552, 81)
Test: (516, 81)
✓ Todas las clases presentes en ambos conjuntos


## 2  Entrenamiento del modelo

### 2.1  Inicializacion de la neural network



*   Es valido cambiar la *semilla_red* para arrancar el entrenamiento con distintas rectas iniciales


In [ ]:
# defino la red multiperceptron
semilla_red = 26011989  # define las rectas iniciales

# Arquitectura: 80 componentes → capa oculta → 14 clases (alumnos)
# hidden_layers_sizes: solo las capas ocultas, NO la capa de salida
# layers_func: una funcion por cada capa incluyendo la de salida
hidden = [20]                        # una capa oculta de 20 neuronas
n_capas = len(hidden) + 1            # ocultas + salida
layers_func = ['tansig', 'logsig']   # tansig en hidden, logsig en salida

mp = multiperceptron()
mp.inicializar(
    df=df_train,
    campos=campos,           # ["x1", "x2", ..., "x80"]
    clase="y",               # columna con el nombre del alumno
    hidden_layers_sizes=hidden.copy(),  # copia porque se muta internamente
    layers_func=layers_func,
    semilla=semilla_red,
    carpeta=carpeta
    )

print(f"Arquitectura: {n_componentes} → {hidden} → {len(clases_train)}")
print(f"Funciones:    {layers_func}")
print(f"Modelo se guardara en: {carpeta}")

Arquitectura: 80 → [20] → 14
Funciones:    ['tansig', 'logsig']
Modelo se guardara en: /content/buckets/b1/nn_isomap/


### 2.2 Entrenamiento de la neural network = backpropagation

Aqui se hace el trabajo pesado de entrenar la red neuronal

Es necesario experimentar con


*   learning_rate
*   lr_momento
*   epoch_limit  y MSE_umbral



In [ ]:
# entreno la neural network con BackPropagation

(epoch, MSE, train_error_rate) = mp.entrenar(
    epoch_limit=1500,
    MSE_umbral=0.018,
    learning_rate=0.05,
    lr_momento=0.05,
    save_frequency=100,
    retomar=True)

epoch    102  |  MSE 0.017964


#### Visualizacion de los resultados de la salida del entrenamiento de la red

In [ ]:
# las metricas basica de la red
print("epoch :", epoch)
print("MSE :", MSE)
print("train_error_rate :", train_error_rate)

epoch : 102
MSE : 0.017964399367955932
train_error_rate : 0.19329896907216496


In [ ]:
# la primera hidden layer
print("W :", mp.red["layer"][0]["W"])
print()
print("w0 :", mp.red["layer"][0]["w0"])

W : [[ 1.93664067 -0.98709777  0.30789377 ... -0.46551815  0.31323278
   0.1403178 ]
 [-0.43473969 -1.08428158  0.52759906 ... -0.78162308 -0.34685681
   0.73764628]
 [-0.04450189  0.01714601  0.32227638 ...  0.49382886 -0.01554064
   0.32654348]
 ...
 [-1.81255171 -0.58264038 -1.19327123 ...  0.30697667  1.33735997
   0.2285793 ]
 [ 0.31668311  0.31029767  0.04933822 ... -0.07406447 -0.10112304
   0.23125273]
 [ 1.35112716 -0.46615942 -1.31477185 ...  0.39340178 -0.23873919
   0.24930232]]

w0 : [[-1.87670684]
 [-0.80357151]
 [ 2.68502091]
 [ 3.22290525]
 [ 1.13004925]
 [ 0.23432364]
 [-1.2258262 ]
 [-0.34736107]
 [-3.72121616]
 [ 0.31590127]
 [ 0.59124117]
 [ 0.01820115]
 [ 0.04084974]
 [-0.03950001]
 [-0.80098245]
 [-1.78296795]
 [ 1.1701739 ]
 [ 1.33545341]
 [-3.52625112]
 [ 0.81245018]]


### 2.3 Entrenamiento en caso de retomar



*   Si se cortó el colab
*   Si quiero extender la corrida a mas epochs
*   Si quiero cambiar el learninh_rate
*   Si quiero cambiar el MSE_umbral



In [ ]:
(epoch, MSE, train_error_rate) = mp.entrenar(
    epoch_limit=1800, # aumento
    MSE_umbral=0.001,
    learning_rate=0.05,
    lr_momento=0.05,
    save_frequency=100,
    retomar=True)

Visualizacion de los resultados de salida de un post entrenamiento

In [ ]:
print("epoch :", epoch)
print("MSE :", MSE)
print("train_error_rate :", train_error_rate)

epoch : 60
MSE : 0.017968774029607185
train_error_rate : 0.17396907216494845


In [ ]:
# la primera hidden layer
print("W :", mp.red["layer"][0]["W"])
print()
print("w0 :", mp.red["layer"][0]["w0"])

W : [[ 0.002026    0.26011049 -0.20584858 ... -0.30722301 -0.0794986
   0.64735488]
 [ 0.32701818 -0.7783148   0.08875319 ...  0.20064543  0.17163484
   0.24611787]
 [-0.48646559  0.38699035  0.71481734 ... -0.03367769  0.48589626
   0.00519156]
 ...
 [ 0.89165039 -0.66038756 -0.72722394 ...  0.15060993 -0.02543308
   0.44314445]
 [ 0.49385252 -0.06429283 -0.18833613 ... -0.14502552 -0.21945669
   0.10955415]
 [ 1.35688141 -0.77691708 -1.2977218  ... -0.03042102 -0.52431835
   0.08875813]]

w0 : [[-1.90551262]
 [-1.38234384]
 [ 2.25721583]
 [ 2.97424122]
 [ 1.73872625]
 [-0.15200314]
 [ 0.94215051]
 [-1.13187815]
 [-1.65012509]
 [ 1.63490378]
 [ 2.07151069]
 [ 0.50220046]
 [ 0.7852234 ]
 [ 0.26973247]
 [-1.95826796]
 [-2.32979751]
 [-0.29810017]
 [ 0.90034515]
 [-3.16703249]
 [ 1.01256361]]


## 3  Prediccion en los datos de Testing


Se muestran los datos de testing, que son distintos a los de training

### 3.1 Prediccion en caliente

In [ ]:
(y_hat, y_raw, test_error_rate) = mp.predecir(df_new=df_test, campos=campos, clase='y')

#### Visualizacion del error en testing

In [ ]:
# RECORDAR: la UNICA metrica valida para decidir es test_error_rate
# Si test_error_rate >> train_error_rate → OVERFITTING
print("error_rate (train, test): ", train_error_rate, test_error_rate)

error_rate (train, test):  0.19329896907216496 0.45542635658914726


#### Visualizacion de la prediccion en testing

In [ ]:
tb_salida_test = pl.DataFrame( {"clase":df_test["y"], "pred":y_hat, "y_raw":y_raw })
tb_salida_test

Loading ITables v2.7.3 from the internet... (need help?)


## 4 Prediccion en datos NUEVOS


*   La red fue entrenada en el pasado, y se grabó al drive
*   Ya no esta disponible la sesion donde se entreno
*   No quiero volver a entrenar de cero

In [ ]:
# cargo datos NUEVOS
# Estos arrays los genera la Celda 12 del notebook de Eigenfaces
# (pipeline: foto nueva → deteccion → recorte → proyeccion PCA/ISOMAP)

if MODO == "pca":
    X_new_arr = np.load(f"{CARPETA_DATOS}/fotos_test_proyectadas_pca.npy")
else:
    X_new_arr = np.load(f"{CARPETA_DATOS}/fotos_test_proyectadas_isomap.npy")

y_new_arr = np.load(f"{CARPETA_DATOS}/etiquetas_test.npy", allow_pickle=True)

data_new = {campos[i]: X_new_arr[:, i].astype(np.float64) for i in range(n_componentes)}
data_new["y"] = y_new_arr.astype(str)
df_new = pl.DataFrame(data_new)

print(f"Fotos nuevas cargadas: {df_new.shape}")
df_new

Fotos nuevas cargadas: (14, 81)


Loading ITables v2.7.3 from the internet... (need help?)


In [ ]:
# cargo modelo grabado y lo aplico a los datos nuevos

mp_frio = multiperceptron()
(epoch, MSE, train_error_rate) = mp_frio.cargar_modelo(carpeta)

(y_hat, y_raw, new_error_rate) = mp_frio.predecir(df_new=df_new, campos=campos, clase='y')

#### Visualizacion del error modeloa aplicado a datos nuevos

In [ ]:
print("error_rate (train, new): ",  train_error_rate, new_error_rate)

error_rate (train, new):  0.17396907216494845 0.35714285714285715


#### Visualizacion de la prediccion en datos nuevos

In [ ]:
tb_salida_new = pl.DataFrame( {"clase":df_new["y"], "pred":y_hat, "y_raw":y_raw })
tb_salida_new

Loading ITables v2.7.3 from the internet... (need help?)


## 5. Umbral de confianza — clasificar como 'desconocido'

Si el score maximo de la neurona de salida es menor a un umbral,
la red no esta segura de la identidad → clasificar como **'desconocido'**.

Esto se usa cuando el profesor suba una foto de alguien que no es del grupo.

In [ ]:
UMBRAL_DESCONOCIDO = 0.70  # experimentar con este valor

# Aplicar a los datos nuevos
y_hat_final = np.where(
    y_raw >= UMBRAL_DESCONOCIDO,
    y_hat,           # confianza alta → mantener prediccion
    "desconocido"    # confianza baja → desconocido
)

tb_final = pl.DataFrame({
    "clase": df_new["y"],
    "pred":  y_hat_final,
    "score": np.round(y_raw, 4)
})
tb_final

Loading ITables v2.7.3 from the internet... (need help?)


### 5.1 Demo del umbral sobre datos de test
Para verificar que funciona antes de tener las fotos nuevas.

In [ ]:
# demo: aplico el umbral sobre las predicciones de test
(y_hat_test, y_raw_test, _) = mp.predecir(df_new=df_test, campos=campos, clase='y')

y_hat_demo = np.where(
    y_raw_test >= UMBRAL_DESCONOCIDO,
    y_hat_test,
    "desconocido"
)

n_desconocidos = np.sum(y_hat_demo == "desconocido")
print(f"Umbral: {UMBRAL_DESCONOCIDO}")
print(f"Fotos clasificadas como 'desconocido': {n_desconocidos} de {len(y_hat_demo)}")

tb_demo = pl.DataFrame({
    "clase":      df_test["y"],
    "pred_final": y_hat_demo,
    "score":      np.round(y_raw_test, 4)
})
tb_demo

Umbral: 0.7
Fotos clasificadas como 'desconocido': 226 de 516


Loading ITables v2.7.3 from the internet... (need help?)
